In [2]:
import pandas as pd
import numpy as np
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# 1. 데이터 로드 (경로 그대로 유지)
# =========================
df = pd.read_excel("../데이터 베이스/2. 12개 입력값 머신러닝 전단파괴만.xlsx")

y = df.iloc[:, -1].values          # Pu (마지막 열)
X = df.iloc[:, :-1].copy()

# ID 컬럼 있으면 제거
for col in ["TestID", "ID", "Name"]:
    if col in X.columns:
        X = X.drop(columns=[col])

# (선택) 변수 매핑 확인용: x0, x1, ...가 어떤 컬럼인지 출력
print("=== X column order (x0, x1, ...) ===")
for i, c in enumerate(X.columns):
    print(f"x{i} = {c}")

X = X.values

# =========================
# 2. Train / Test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 3. PySR 설정
# =========================
model = PySRRegressor(
    niterations=2000,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square"],     # x² 허용
    maxsize=15,                     # 식 복잡도 제한
    elementwise_loss="(x - y)^2",   # 최신 옵션 (loss 경고 방지)
    model_selection="best",
    verbosity=1,
    random_state=42,
    deterministic=True,             # 재현성
    parallelism="serial",           # 재현성
)

# =========================
# 4. 학습
# =========================
model.fit(X_train, y_train)

# =========================
# 5. 예측 성능 (R², RMSE, MAE)
# =========================
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)

# sklearn 구버전 호환: squared=False 대신 sqrt(MSE)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

mae = mean_absolute_error(y_test, y_pred)

print("\n=== Test performance ===")
print("R²  =", r2)
print("RMSE =", rmse)
print("MAE  =", mae)

# =========================
# 6. 최종 경험식
# =========================
print("\n=== Best equation (model_selection='best') ===")
print(model)

# =========================
# 7. 후보식 테이블(복잡도별 Hall of Fame)
# =========================
print("\n=== Hall of Fame equations table ===")
print(model.equations_)


=== X column order (x0, x1, ...) ===
x0 = id
x1 = d
x2 = h
x3 = b_0
x4 = b
x5 = A
x6 = b_h_col
x7 = fc
x8 = fy
x9 = r
x10 = a_d
x11 = fy_fc
x12 = b_d


c:\Users\SSC-3\Desktop\code\Punching-shear\venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


JuliaError: UndefVarError: `x` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Stacktrace:
 [1] top-level scope
   @ none:1
 [2] eval
   @ .\boot.jl:430 [inlined]
 [3] eval
   @ .\Base.jl:130 [inlined]
 [4] pyjlmodule_seval(self::Module, expr::Py)
   @ PythonCall.JlWrap C:\Users\SSC-3\.julia\packages\PythonCall\avYrV\src\JlWrap\module.jl:13
 [5] _pyjl_callmethod(f::Any, self_::Ptr{PythonCall.C.PyObject}, args_::Ptr{PythonCall.C.PyObject}, nargs::Int64)
   @ PythonCall.JlWrap C:\Users\SSC-3\.julia\packages\PythonCall\avYrV\src\JlWrap\base.jl:67
 [6] _pyjl_callmethod(o::Ptr{PythonCall.C.PyObject}, args::Ptr{PythonCall.C.PyObject})
   @ PythonCall.JlWrap.Cjl C:\Users\SSC-3\.julia\packages\PythonCall\avYrV\src\JlWrap\C.jl:63

In [1]:
import pandas as pd
import numpy as np
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# 1) 데이터 로드 (경로 그대로)
# =========================
df = pd.read_excel("../데이터 베이스/2. 12개 입력값 머신러닝 전단파괴만.xlsx")

y = df.iloc[:, -1].astype(float).values          # Vn
Xdf = df.iloc[:, :-1].copy()

# ✅ id 제거 (너 파일은 id가 입력에 들어가 있었음)
for col in ["id", "TestID", "ID", "Name"]:
    if col in Xdf.columns:
        Xdf = Xdf.drop(columns=[col])

# ✅ float 강제
X = Xdf.astype(float).values

# (선택) x0, x1 매핑 확인
print("=== Variable index mapping (x0, x1, ...) ===")
for i, c in enumerate(Xdf.columns):
    print(f"x{i} = {c}")

# =========================
# 2) Train / Test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 3) PySR 설정
# =========================
model = PySRRegressor(
    niterations=2000,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square"],
    maxsize=15,
    model_selection="best",
    verbosity=2,
    deterministic=True,
    parallelism="serial",
    random_state=42,
    variable_names=list(Xdf.columns),
)

# =========================
# 4) 학습
# =========================
model.fit(X_train, y_train)

# =========================
# 5) 성능 (R² / RMSE / MAE)
# =========================
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))  # sklearn 호환
mae = mean_absolute_error(y_test, y_pred)

print("\n=== Test performance ===")
print("R²  =", r2)
print("RMSE =", rmse)
print("MAE  =", mae)

# =========================
# 6) 최종 경험식
# =========================
print("\n=== Best equation ===")
print(model)

print("\n=== Hall of Fame ===")
print(model.equations_)


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


c:\Users\SSC-3\Desktop\code\Punching-shear\venv\Lib\site-packages\pysr\sr.py:1046: FutureWarning: `variable_names` is a data-dependent parameter and should be passed when fit is called. Ignoring parameter; please pass `variable_names` during the call to fit instead.
  warnings.warn(
c:\Users\SSC-3\Desktop\code\Punching-shear\venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...


=== Variable index mapping (x0, x1, ...) ===
x0 = d
x1 = h
x2 = b_0
x3 = b
x4 = A
x5 = b_h_col
x6 = fc
x7 = fy
x8 = r
x9 = a_d
x10 = fy_fc
x11 = b_d


[ Info: Started!



Expressions evaluated per second: 3.230e+05
Progress: 1571 / 62000 total iterations (2.534%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.994e+05  0.000e+00  y = 357.39
3           6.134e+04  5.832e-01  y = x₀ * 4.1099
4           5.235e+04  1.539e-01  y = square(x₁ * -0.10717)
5           3.594e+04  3.705e-01  y = (x₀ + -49.448) * 6.1953
6           2.469e+04  3.724e-01  y = square((x₁ + x₆) * -0.098193)
8           1.514e+04  2.436e-01  y = square((x₈ - -0.088547) * (x₆ + x₁))
10          1.323e+04  6.717e-02  y = square(x₁₁ + ((x₈ - -0.085247) * (x₆ + x₁)))
12          1.288e+04  1.282e-02  y = square(((x₈ - -0.08641) * (x₆ + x₁)) + x₁₁) + -22.563
14          1.282e+04  1.748e-03  y = square((x₉ * -0.10861) + (x₁₁ + ((x₁ + x₆) * (x₈ - -0....
                               

[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.919e+05  0.000e+00  y = x₃
3           6.134e+04  5.641e-01  y = x₀ * 4.1099
4           5.235e+04  1.539e-01  y = square(x₁ * -0.10717)
5           3.594e+04  3.705e-01  y = (x₀ * 6.1953) + -306.35
6           2.469e+04  3.724e-01  y = square((x₆ + x₁) * 0.098191)
7           2.069e+04  1.743e-01  y = x₀ * ((x₀ * x₈) + 2.0962)
8           1.513e+04  3.118e-01  y = square((x₆ + x₁) * (x₈ + 0.088866))
9           1.501e+04  7.405e-03  y = x₃ + ((x₀ * 0.00029985) * (x₆ * x₀))
10          1.323e+04  1.251e-01  y = square(((x₈ - -0.085246) * (x₁ + x₆)) + x₁₁)
11          1.248e+04  5.685e-02  y = ((x₀ * x₈) + (x₇ * 0.0053003)) * (x₀ - x₁₀)
12          1.170e+04  6.362e-02  y = (x₄ * 0.0010012) + square((x₆ + x₁) * (x₈ - -0.084719)...
                                      )
13          9.709e+03  1.854e-01  y = (x₃ + (((x₈ + (x₆ * 0.000

In [2]:
# =========================
# 7) Hall of Fame 각 경험식별 성능 (R²/RMSE/MAE)
# =========================
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

eqs = model.equations_.copy()

results = []

# PySR 버전에 따라 equation 인자를 받는지/이름이 다른지 달라서 안전하게 처리
for idx, row in eqs.iterrows():
    # equation 컬럼명은 보통 'equation' 또는 'Equation'로 들어있음
    eq_str = row.get("equation", None)
    if eq_str is None:
        eq_str = row.get("Equation", None)

    # sympy_format(있으면 더 안정적)
    sympy_eq = row.get("sympy_format", None)

    # 1) 가능하면 sympy_format을 equation으로 쓰고
    # 2) 안 되면 equation 문자열 사용
    eq_to_use = sympy_eq if sympy_eq is not None else eq_str

    try:
        # PySR 일부 버전: predict(X, equation=...) 지원
        yhat = model.predict(X_test, equation=eq_to_use)
    except TypeError:
        try:
            # PySR 일부 버전: predict(X, index=...) 지원 (Hall of Fame 인덱스)
            yhat = model.predict(X_test, index=idx)
        except Exception:
            # 최후 수단: 해당 식은 스킵
            results.append({
                "hof_index": idx,
                "complexity": row.get("complexity", row.get("Complexity", None)),
                "loss_train": row.get("loss", row.get("Loss", None)),
                "equation": str(eq_str),
                "R2_test": np.nan,
                "RMSE_test": np.nan,
                "MAE_test": np.nan,
                "status": "FAILED_PREDICT"
            })
            continue

    r2_i = r2_score(y_test, yhat)
    rmse_i = np.sqrt(mean_squared_error(y_test, yhat))
    mae_i = mean_absolute_error(y_test, yhat)

    results.append({
        "hof_index": idx,
        "complexity": row.get("complexity", row.get("Complexity", None)),
        "loss_train": row.get("loss", row.get("Loss", None)),
        "equation": str(eq_str),
        "R2_test": r2_i,
        "RMSE_test": rmse_i,
        "MAE_test": mae_i,
        "status": "OK"
    })

perf_df = pd.DataFrame(results)

# R² 높은 순으로 정렬
perf_df = perf_df.sort_values(by="R2_test", ascending=False)

print("\n=== Hall of Fame performance on TEST set (sorted by R²) ===")
print(perf_df[["hof_index","complexity","loss_train","R2_test","RMSE_test","MAE_test","status","equation"]].to_string(index=False))

# (선택) 엑셀로 저장
perf_df.to_excel("pysr_hof_performance.xlsx", index=False)
print("\nSaved: pysr_hof_performance.xlsx")



=== Hall of Fame performance on TEST set (sorted by R²) ===
 hof_index  complexity  loss_train   R2_test  RMSE_test   MAE_test status                                                                     equation
        10          12   11700.418  0.924063  72.012102  54.346743     OK                  (x4 * 0.0010011537) + square((x6 + x1) * (x8 - -0.0847188))
        12          15    9129.844  0.910371  78.235401  61.050637     OK ((x0 * (((x8 + (x6 * 0.00030549863)) * x0) - -0.90551406)) + x3) / 1.7080495
        11          13    9708.706  0.898554  83.233149  64.286144     OK                 (x3 + (((x8 + (x6 * 0.00024757723)) * x0) * x0)) / 1.3910532
         8          10   13225.757  0.891738  85.983916  64.779113     OK                              square(((x8 - -0.085246414) * (x1 + x6)) + x11)
         6           8   15131.552  0.871831  93.555791  61.959872     OK                                         square((x6 + x1) * (x8 + 0.0888656))
         9          11   12481.92

In [5]:
import pandas as pd
import numpy as np
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =========================
# 1) 데이터 로드
# =========================
# 너 PC면 여기만 로컬 경로로 바꿔줘
df = pd.read_excel("../데이터 베이스/2. 12개 입력값 머신러닝 전단파괴만.xlsx")

# =========================
# 2) 마지막 열 = 타겟(Vn)
# =========================
y = pd.to_numeric(df.iloc[:, -1], errors="raise").astype(float).values

# =========================
# 3) 지정한 변수만 사용하도록 강제 선택
#   사용 변수(요청):
#   a/d_avg, b/d_avg, fy/fc', b0, b, A, d_avg, fy, ρ, Shape
#
#   (네 파일 컬럼명과 매핑)
#   a/d_avg  -> a_d
#   b/d_avg  -> b_d
#   fy/fc'   -> fy_fc
#   b0       -> b_0
#   ρ        -> r
#   d_avg    -> d   (네 데이터에 d_avg가 따로 있으면 그걸로 바꿔)
# =========================
col_map = {
    "a/d_avg": "a_d",
    "b/d_avg": "b_d",
    "fy/fc'":  "fy_fc",
    "b0":      "b_0",
    "b":       "b",
    "A":       "A",
    "d_avg":   "d",
    "fy":      "fy",
    "ρ":       "r",
    "Shape":   "Shape",  # 파일에 없으면 자동으로 제외됨
}

# 실제 파일에 존재하는 컬럼만 선택
use_cols = []
var_names = []  # PySR에 보여줄 변수명(논문 표기 유지)
missing = []

for nice_name, real_col in col_map.items():
    if real_col in df.columns:
        use_cols.append(real_col)
        var_names.append(nice_name)
    else:
        missing.append((nice_name, real_col))

print("=== 선택된 입력 변수 ===")
for nn, rc in zip(var_names, use_cols):
    print(f"- {nn}  <-  {rc}")

if missing:
    print("\n[경고] 파일에 없어서 제외된 변수:")
    for nn, rc in missing:
        print(f"- {nn} (찾은 컬럼명: {rc})")

# X 구성 (지정 변수만!)
Xdf = df[use_cols].copy()

# 숫자형 강제 (Shape가 문자면 여기서 에러 or NaN 될 수 있음)
# Shape가 범주형(문자)이라면, 아래에서 자동 인코딩하도록 처리해줌.
for c in Xdf.columns:
    if Xdf[c].dtype == "object":
        # 범주형 -> 정수 인코딩 (PySR은 숫자만 받음)
        Xdf[c] = Xdf[c].astype("category").cat.codes

X = Xdf.astype(float).values

# 안전 체크
assert not np.isnan(X).any(), "X에 NaN이 있습니다."
assert not np.isinf(X).any(), "X에 Inf가 있습니다."
assert not np.isnan(y).any(), "y에 NaN이 있습니다."
assert not np.isinf(y).any(), "y에 Inf가 있습니다."

# =========================
# 4) Train/Test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 5) PySR 설정 (지정 변수 조합만 탐색)
# =========================
model = PySRRegressor(
    niterations=2000,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["square"],      # 필요하면 ["square","sqrt","log","exp"] 등으로 확장 가능
    maxsize=20,                      # 식 복잡도(올리면 성능↑ 가능, 과적합↑)
    model_selection="best",
    verbosity=2,                     # 진행 로그 보이게
    deterministic=True,
    parallelism="serial",
    random_state=42,
    variable_names=var_names,        # ✅ 네가 지정한 변수명으로 식 출력
    # elementwise_loss="loss(x, y) = (x - y)^2",  # 써도 됨(형식 중요). 기본도 MSE라 생략 가능
)

# =========================
# 6) 학습
# =========================
model.fit(X_train, y_train)

# =========================
# 7) 성능 출력
# =========================
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))  # sklearn 호환
mae = mean_absolute_error(y_test, y_pred)

print("\n=== Test performance ===")
print("R²  =", r2)
print("RMSE =", rmse)
print("MAE  =", mae)

# =========================
# 8) 최종 경험식 + 후보식 테이블
# =========================
print("\n=== Best equation ===")
print(model)

print("\n=== Hall of Fame (candidate equations) ===")
print(model.equations_)


=== 선택된 입력 변수 ===
- a/d_avg  <-  a_d
- b/d_avg  <-  b_d
- fy/fc'  <-  fy_fc
- b0  <-  b_0
- b  <-  b
- A  <-  A
- d_avg  <-  d
- fy  <-  fy
- ρ  <-  r

[경고] 파일에 없어서 제외된 변수:
- Shape (찾은 컬럼명: Shape)


c:\Users\SSC-3\Desktop\code\Punching-shear\venv\Lib\site-packages\pysr\sr.py:1046: FutureWarning: `variable_names` is a data-dependent parameter and should be passed when fit is called. Ignoring parameter; please pass `variable_names` during the call to fit instead.
  warnings.warn(
c:\Users\SSC-3\Desktop\code\Punching-shear\venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 3.010e+05
Progress: 1360 / 62000 total iterations (2.194%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.919e+05  0.000e+00  y = x₄
3           6.134e+04  5.641e-01  y = x₆ * 4.1099
5           3.594e+04  2.640e-01  y = (x₆ + -49.443) * 6.1948
6           3.496e+04  2.344e-02  y = x₄ + (square(x₆) * x₈)
7           2.069e+04  5.211e-01  y = ((x₆ * x₈) + 2.0962) * x₆
9           1.646e+04  1.131e-01  y = ((x₆ * x₈) - -3.0972) * (x₆ + -31.988)
11          1.298e+04  1.184e-01  y = (((x₇ * x₆) * (x₈ * 0.0021205)) + 2.0399) * x₆
12          1.298e+04  -0.000e+00  y = (x₆ * 2.04) + ((x₇ * (square(x₆) * 0.0021204)) * x₈)
13          1.075e+04  1.874e-01  y = ((x₆ * (x₇ * (x₈ * 0.0020493))) + 2.8713) * (x₆ + -26....
                                  

[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 2.740e+05
Progress: 61143 / 62000 total iterations (98.618%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.919e+05  0.000e+00  y = x₄
3           6.134e+04  5.641e-01  y = x₆ * 4.1099
5           3.594e+04  2.640e-01  y = (x₆ * 6.1953) + -306.35
6           3.190e+04  1.169e-01  y = (x₈ + 0.0066314) * square(x₆)
7           2.069e+04  4.297e-01  y = ((x₆ * x₈) + 2.0962) * x₆
8           1.971e+04  4.659e-02  y = ((x₈ + 0.0038848) * square(x₆)) + x₆
9           1.575e+04  2.232e-01  y = ((x₇ * 0.0044226) + (x₆ * x₈)) * x₆
10          1.552e+04  1.403e-02  y = x₆ + ((x₈ + (x₁ * 0.0041656)) * square(x₆))
11          1.248e+04  2.163e-01  y = (x₆ - x₂) * ((x₇ * 0.0053006) + (x₈ * x₆))
13          1.075e+04  7.427e-02  y = (((x₈ * (x₇ * 0.002049)) 

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# model, X_train, X_test, y_train, y_test 가 이미 위에서 생성돼 있어야 함

eqdf = model.equations_.copy()

rows = []
for i, row in eqdf.iterrows():
    eq_str = row["equation"]
    comp = row["complexity"]
    loss = row["loss"]

    # ✅ 각 경험식별로 predict
    # 가장 안정적인 방법: lambda_format 사용
    f = row["lambda_format"]  # PySRFunction(X=>...)
    try:
        # PySRFunction은 호출 시 X를 줘야 함 (보통 numpy array)
        yhat_train = f(X_train)
        yhat_test  = f(X_test)
    except Exception:
        # 혹시 lambda_format이 직접 callable이 아닌 환경이면 fallback
        # (대부분은 위에서 바로 됨)
        yhat_train = model.predict(X_train, equation=row["sympy_format"])
        yhat_test  = model.predict(X_test, equation=row["sympy_format"])

    r2_tr = r2_score(y_train, yhat_train)
    r2_te = r2_score(y_test,  yhat_test)

    rmse_tr = np.sqrt(mean_squared_error(y_train, yhat_train))
    rmse_te = np.sqrt(mean_squared_error(y_test,  yhat_test))

    mae_tr = mean_absolute_error(y_train, yhat_train)
    mae_te = mean_absolute_error(y_test,  yhat_test)

    rows.append({
        "hof_idx": i,
        "complexity": comp,
        "loss(train_in_PySR)": loss,
        "R2_train": r2_tr,
        "R2_test":  r2_te,
        "RMSE_test": rmse_te,
        "MAE_test":  mae_te,
        "equation": eq_str
    })

perf = pd.DataFrame(rows)

# ✅ test R² 높은 순으로 정렬해서 보기
perf_sorted = perf.sort_values("R2_test", ascending=False).reset_index(drop=True)

print("=== Hall of Fame equations performance (sorted by Test R²) ===")
display(perf_sorted)

# (선택) 엑셀 저장
perf_sorted.to_excel("pysr_hof_R2_table.xlsx", index=False)
print("Saved: pysr_hof_R2_table.xlsx")


=== Hall of Fame equations performance (sorted by Test R²) ===


,hof_idx,complexity,loss(train_in_PySR),R2_train,R2_test,RMSE_test,MAE_test,equation
0,11,17,8994.048,0.954900,0.899243,82.950129,61.540759,(x6 - ((x2 + 22.222042) / x1)) * (((x6 * (x7 *...
1,10,15,9308.633,0.953323,0.893830,85.149177,64.090873,(x6 - x2) * (((x6 * 0.0019101257) * (x1 + (x7 ...
2,12,19,8470.944,0.957523,0.889624,86.819579,62.494815,((x6 - (x2 / x1)) - 19.338255) * (((((x8 * x6)...
3,7,10,15515.056,0.922201,0.850202,101.142122,73.891064,x6 + ((x8 + (x1 * 0.0041656317)) * square(x6))
4,8,11,12481.923,0.937410,0.848089,101.853179,75.270157,(x6 - x2) * ((x7 * 0.0053005624) + (x8 * x6))
5,9,13,10746.838,0.946111,0.846229,102.474596,73.043977,(((x8 * (x7 * 0.002048973)) * x6) + 2.872118) ...
6,4,7,20694.600,0.896228,0.823629,109.747228,84.884414,((x6 * x8) + 2.0962331) * x6
7,6,9,15749.879,0.921023,0.816880,111.827276,81.443589,((x7 * 0.004422583) + (x6 * x8)) * x6
8,5,8,19712.158,0.901155,0.814878,112.436792,79.812690,((x8 + 0.0038847562) * square(x6)) + x6
9,3,6,31899.588,0.840042,0.652152,154.125558,117.290317,(x8 + 0.0066313846) * square(x6)


Saved: pysr_hof_R2_table.xlsx


In [13]:
import re
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# -----------------------------
# 1) 변수 매핑 (확실하게)
# -----------------------------
# var_names 는 위에서 PySR에 넘긴 리스트
# 예: ['a/d_avg','b/d_avg','fy/fc\'','b0','b','A','d_avg','fy','ρ']

var_map = {f"x{i}": var_names[i] for i in range(len(var_names))}

eqdf = model.equations_.copy()

rows = []

for i, row in eqdf.iterrows():

    eq_raw = row["equation"]
    comp   = row["complexity"]

    # -----------------------------
    # 2) x숫자 → 실제 변수명 변환 (정규식)
    # -----------------------------
    eq_real = eq_raw
    for k, v in var_map.items():
        eq_real = re.sub(rf"\b{k}\b", v, eq_real)

    # -----------------------------
    # 3) 성능 계산
    # -----------------------------
    f = row["lambda_format"]

    yhat_train = f(X_train)
    yhat_test  = f(X_test)

    r2_tr = r2_score(y_train, yhat_train)
    r2_te = r2_score(y_test, yhat_test)

    rmse_te = np.sqrt(mean_squared_error(y_test, yhat_test))
    mae_te  = mean_absolute_error(y_test, yhat_test)

    rows.append({
        "complexity": comp,
        "R2_test":  r2_te,
        "RMSE_test": rmse_te,
        "MAE_test":  mae_te,
        "equation_real_symbol": eq_real
    })

perf = pd.DataFrame(rows)
perf = perf.sort_values("R2_test", ascending=False).reset_index(drop=True)
perf.to_excel("pysr_hof_R2_table.xlsx", index=False)
display(perf)


,complexity,R2_test,RMSE_test,MAE_test,equation_real_symbol
0,17,0.899243,82.950129,61.540759,(d_avg - ((fy/fc' + 22.222042) / b/d_avg)) * (...
1,15,0.893830,85.149177,64.090873,(d_avg - fy/fc') * (((d_avg * 0.0019101257) * ...
2,19,0.889624,86.819579,62.494815,((d_avg - (fy/fc' / b/d_avg)) - 19.338255) * (...
3,10,0.850202,101.142122,73.891064,d_avg + ((ρ + (b/d_avg * 0.0041656317)) * squa...
4,11,0.848089,101.853179,75.270157,(d_avg - fy/fc') * ((fy * 0.0053005624) + (ρ *...
5,13,0.846229,102.474596,73.043977,(((ρ * (fy * 0.002048973)) * d_avg) + 2.872118...
6,7,0.823629,109.747228,84.884414,((d_avg * ρ) + 2.0962331) * d_avg
7,9,0.816880,111.827276,81.443589,((fy * 0.004422583) + (d_avg * ρ)) * d_avg
8,8,0.814878,112.436792,79.812690,((ρ + 0.0038847562) * square(d_avg)) + d_avg
9,6,0.652152,154.125558,117.290317,(ρ + 0.0066313846) * square(d_avg)
